# ww6 강화학습 Project 과제

# ※ 실험(Algorithm 구현): 한화 이글스 최대 기대 득점 타순 구성하기
   1. env: 1인닝 3out
   2. Raw Data: KBO 환화 이글스 야수 엔트리 및 아웃, 병살, 희생 번트, 희생플라이, BB, H, 2B, 3B, HR 확률, CSV 생성 Load

   3. agent: 감독
   
   4. 목표: 최대 기대값의 타순(정책) 추정
   
   5. State: 타순, 아웃, 1B, 2B, 3B, 점수
   
   6. Action: 타순에 타자 중  선택
   
   7. Next state 업데이트: 선수 Stats에 따른 Play 결과 확률에 따른 업데이트

   8. Reward: 업데이트된 R값

   9. Terminate: 3out

   10. Output: Least-Squares_Temporal_Difference_Learning으로 추정한 최적 정책 및 그 시각화

In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import matplotlib.pyplot as plt
plt.rc('font', family='NanumBarunGothic')

In [ ]:
import pandas as pd
import numpy as np
from collections import deque
import matplotlib.pyplot as plt
import time
import random

class BaseballEnv: # 야구 게임 환경 구성
    def __init__(self, stats_csv):
        self.stats = pd.read_csv(stats_csv) # CSV 데이터 파일 읽기
        self.stats.columns = self.stats.columns.str.strip() # 파일 내 한글 컬럼 명 공백 제거
        self.stats['player'] = self.stats['player'].str.strip() # 플레이어 이름에 공백에 의한 에러 방지
        self.stats.set_index('player', inplace=True) # 인덱스 설정
        self.result_mapping = {
            "아웃": "Out",
            "병살타": "DoublePlay",
            "희생번트": "SacrificeBunt",
            "희생플라이": "SacrificeFly",
            "BB": "BB",
            "H": "H",
            "2B": "2B",
            "3B": "3B",
            "HR": "HR"
        } # Action(선수 선택)의 결과 맵핑 한글 영문화
        self.reset() # 초기 State reset:아웃, 주자, 점수

    def _flatten(self, x): # 튜플로 변환
        if isinstance(x, list):
            return tuple(self._flatten(i) for i in x)
        return x

    def _get_state(self): # state 호출: 아웃, 주자(튜플), 점수
        return (self.outs, tuple(self._flatten(self.runners)), self.score)

    def reset(self):
        self.outs = 0 # 아웃 초기화
        self.runners = [0, 0, 0]  # 1B, 2B, 3B 초기화
        self.score = 0 # 점수 초기화
        return self._get_state()

    def step(self, player): # Action(선수 선택)의 결과 갱신
        row = self.stats.loc[player] # 선택 타자의 액션 확률
        probs = row.to_dict() # 딕셔너리 변환
        outcome_kor = np.random.choice(list(probs.keys()), p=list(probs.values())).strip()
        outcome = self.result_mapping[outcome_kor] # play 결과 영어로 맵핑
        reward, out, runners = self._simulate_play(outcome) # play 결과를 업데이트
        self.score += reward # 점수 누적
        self.outs += out # 아웃 누적
        self.runners = runners # 주자
        done = self.outs >= 3 # 인닝 종료
        return self._get_state(), reward, done # 아웃, 주자(튜플), 점수, 보상, 종류 여부

    def _simulate_play(self, result):
        if result == 'Out': # reward(기대 점수) 0, 아웃 1, 주자 유지
          return 0, 1, self.runners
        elif result == 'DoublePlay': # 아웃 2,  진루 0, 병살 1, 임의로 1루 -> 3루 -> 2루 우선 순위로 주자 소거
          return self._advance_runners(2, 0, 1)
        elif result in ['SacrificeBunt', 'SacrificeFly']: # 아웃 1,  진루 1, 병살 0
          return self._advance_runners(1, 1, 0)
        elif result == 'BB': # 아웃 0,  진루 1, 병살 0, 볼넷
          return 0, 0, self._advance_runners(0, 1, 0, walk=True)
        elif result == 'H': # 아웃 0,  진루 1*2, 병살 0
          return self._advance_runners(0, 1, 0)
        elif result == '2B': # 아웃 0,  진루 2*2, 병살 0
          return self._advance_runners(0, 2, 0)
        elif result == '3B': # 아웃 0,  진루 3*2, 병살 0
          return self._advance_runners(0, 3, 0)
        elif result == 'HR': # 아웃 0,  진루 4*2, 병살 0
          return self._advance_runners(0, 3, 0)
        return 0, 0, self.runners

    def _advance_runners(self, out, bases, double, walk=False):
        runners = [0, 0 ,0]
        new_score = 0

        if walk: # 볼넷
          if self.runners[0]:  # 1루 주자가 있으면
            runners[1] = 1 # 2루로 주자 갱신
            if self.runners[1]: # 2루 주자가 있으면
              runners[2] = 1 # 3루로 주자 갱신
              if self.runners[2]: #3루 주가가 있으면
                new_score += 1 # 스코어 1점 갱신
          runners[0] = 1 # 1루는 무조건 1로 갱신
        elif double: # 병살
          for i in [0,2,1]: # 주자 소거 우선 순위
            if self.runners[i]: # 주자 확인해서 있으면
               runners[i] = 0 # 주자 소거
               break
        else: # 진루타 ro 안타
            for i in range(2, -1, -1): # 모든 주자
                if self.runners[i]: # 각 주자
                    if i + bases*2 >= 3: # 3루(2) 보다 크면
                        new_score += 1 # 주자 득점 업데이트
                    else:
                        runners[i + bases*2] = 1 #  주자 안타 별 진루*2
            if bases < 4:
                runners[bases - 1] = 1 # 안타의 경우 타자 -> 주자 업데이트
            else:
                new_score += 1 # 홈런 타자 득점
            runners[0] -= out
        return new_score, out, tuple(runners) # 득점, 아웃, 주자 갱신 반환



In [ ]:
class TDAgent:
    def __init__(self, players, alpha=0.01, gamma=0.9, feature_dim=100):
        self.players = players
        self.alpha = alpha
        self.gamma = gamma
        self.feature_dim = feature_dim
        self.theta = np.zeros((feature_dim, 1))
        self.policy = []

    def featurize(self, state, action):
        feature = np.zeros((self.feature_dim, 1))
        hashable_state = tuple(
            tuple(s) if isinstance(s, list) else s for s in state
        )
        index = hash((hashable_state, action)) % self.feature_dim
        feature[index] = 1
        return feature

    def update(self, s, a, r, s_prime, a_prime):
        phi = self.featurize(s, a)
        phi_prime = self.featurize(s_prime, a_prime)
        # item()을 사용하여 단일 요소 추출
        td_error = r + self.gamma * (phi_prime.T @ self.theta).item() - (phi.T @ self.theta).item()
        self.theta += self.alpha * td_error * phi

    def compute_policy(self):
        values = {}
        for p in self.players:
            s = (0, (0, 0, 0), 0)
            phi = self.featurize(s, p)
            # item()을 사용하여 단일 요소 추출
            values[p] = (phi.T @ self.theta).item()
        sorted_players = sorted(values.items(), key=lambda x: -x[1])
        self.policy = sorted_players[:9]
        return self.policy

In [ ]:
class LSTDAgent:
    def __init__(self, players, gamma=0.9, feature_dim=100):
        self.players = players # 선수 목록
        self.gamma = gamma # 할인율 0.9
        self.feature_dim = feature_dim # 벡터 차원 No
        self.A = np.zeros((feature_dim, feature_dim)) # A = Dim x Dim 0벡터 생성
        self.b = np.zeros((feature_dim, 1)) # b = Dim x 1 0벡터 생성
        self.theta = np.zeros((feature_dim, 1)) # theta = Dim x 1 0벡터 생성
        self.policy = [] # 정책

    def featurize(self, state, action):
        feature = np.zeros((self.feature_dim, 1))
        hashable_state = tuple(
            tuple(s) if isinstance(s, list) else s for s in state
        )
        index = hash((hashable_state, action)) % self.feature_dim
        feature[index] = 1
        return feature

    def update(self, s, a, r, s_prime):
        for next_action in self.players:
            phi = self.featurize(s, a)
            phi_prime = self.featurize(s_prime, next_action)
            delta = phi - self.gamma * phi_prime
            self.A += phi @ delta.T
            self.b += phi * r

    def compute_policy(self):
        self.theta = np.linalg.pinv(self.A) @ self.b # 최소제곱법 (β = A⁻¹ b)
        values = {}
        for p in self.players:
            s = (0, (0, 0, 0), 0)
            phi = self.featurize(s, p)
            values[p] = (phi.T @ self.theta).item()
        sorted_players = sorted(values.items(), key=lambda x: -x[1])
        self.policy = sorted_players[:9]
        return self.policy

In [ ]:
def hash_state_action(state, action):
    hashable_state = tuple(
        tuple(s) if isinstance(s, list) else s for s in state
    )
    return hash((hashable_state, action))

class QLearningAgent:
    def __init__(self, actions, alpha=0.1, gamma=0.9, epsilon=1.0, epsilon_decay=0.99, epsilon_min=0.01):
        self.q_table = {}
        self.actions = actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

    def get_q(self, state, action):
        key = hash_state_action(state, action)
        return self.q_table.get(key, 0.0)

    def update(self, state, action, reward, next_state):
        key = hash_state_action(state, action)
        max_next_q = max([self.get_q(next_state, a) for a in self.actions])
        self.q_table[key] = self.get_q(state, action) + self.alpha * (reward + self.gamma * max_next_q - self.get_q(state, action))

    def select_action(self, state, used_players):
        available_actions = [a for a in self.actions if a not in used_players]
        if not available_actions:
            return random.choice(self.actions)
        if np.random.rand() < self.epsilon:
            return random.choice(available_actions)
        q_values = [self.get_q(state, a) for a in available_actions]
        return available_actions[np.argmax(q_values)]

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon * self.epsilon_decay, self.epsilon_min)

    def compute_policy(self):
        policy = []
        state = (0, (0, 0, 0), 0)
        for _ in range(9):
            best_player = None
            best_q_value = -float('inf')
            for player in self.actions:
                if player not in [p[0] for p in policy]:
                    q_value = self.get_q(state, player)
                    if q_value > best_q_value:
                        best_q_value = q_value
                        best_player = player
            policy.append((best_player, best_q_value))

        return policy

In [ ]:
def calculate_expected_runs(policy, env, num_simulations=1000):
    total_runs = 0
    for _ in range(num_simulations): # 설정 시뮬레이션 횟수 반복
        env.reset() # 초기화
        outs = 0 # 아웃
        player_index = 0 # 1번 타자 부터 시작
        while outs < 3: # 3아웃이 되기 전까지 무한 반복
            player = policy[player_index % len(policy)][0]  # 타자: 3 아웃이 되지 않으면 다시 1번 타자 부터 반복
            _, reward, _ = env.step(player) # 현재 타자의 결과 중 득점
            total_runs += reward # 총 득점에 누적
            outs += env._get_state()[0]  # 현 스테이트의 아웃 결과 업데이트
            player_index += 1 # 다음 타순

    return total_runs / num_simulations # 정책(타순)으로 1인닝을 1000회 진행 시 평균 기대 득점

In [ ]:
if __name__ == "__main__":
    env = BaseballEnv("/content/drive/MyDrive/common/hanwha_stats.csv")
    players = env.stats.index.tolist()

    agent_types = ["LSTD", "TD", "Q"]
    scores = {agent_type: [] for agent_type in agent_types}

    for agent_type in agent_types:
        if agent_type == "TD":
            agent = TDAgent(players)
        elif agent_type == "Q":
            agent = QLearningAgent(players)
        else:
            agent = LSTDAgent(players)

        start = time.time()

        for episode in range(1000):
            state = env.reset()
            used_players = set()
            total_reward = 0

            while True:
                if agent_type == "Q":
                    player = agent.select_action(state, used_players)
                else:
                    for i, player in enumerate(players):
                        if player in used_players:
                            continue
                        break
                next_state, reward, done = env.step(player)
                used_players.add(player)

                if agent_type == "TD":
                    a_prime = None
                    for future_p in players:
                        if future_p not in used_players:
                            a_prime = future_p
                            break
                    if a_prime is None:
                        a_prime = player
                    agent.update(state, player, reward, next_state, a_prime)
                else:
                    agent.update(state, player, reward, next_state)

                state = next_state
                total_reward += reward
                if done or len(used_players) == len(players):
                    break

            if agent_type == "Q":
                agent.decay_epsilon()

            scores[agent_type].append(total_reward)

        # for 루프 외부로 이동
        policy = agent.compute_policy()
        print("\n최적 타순 (예상 가치 기준):")
        for i, (p, v) in enumerate(policy, 1):
            print(f"{i}. {p} (Value: {v:.4f})")

        print(f"\n{agent_type} 계산 소요 시간 ", time.time()-start, '초 소요') # agent_t -> agent_type으로 수정

        expected_runs = calculate_expected_runs(policy, env)
        print(f"\n1이닝 최대 기대 득점: {expected_runs:.4f}")

        # 시각화
        names, values = zip(*policy)
        plt.barh(names[::-1], values[::-1])
        plt.title(f"{agent_type} 기반 예상 가치 순위 (Best 9)") # agent_t -> agent_type으로 수정
        plt.xlabel("Value")
        plt.tight_layout()
        plt.show()

    # 시각화 (학습 에피소드 별 평균 점수 비교)
    plt.figure(figsize=(10, 6))
    for agent_type in agent_types:
        plt.plot(scores[agent_type], label=agent_type, linewidth=2)

    plt.xlabel("Episode", fontsize=12)
    plt.ylabel("Total Score", fontsize=12)
    plt.title("강화학습 Agent별 타순 최적화 점수 비교", fontsize=14)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
# <끝>